# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all entities (RecordSets, Fields, etc.) by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Each entity will be referenced by its Croissant `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL (JSON-LD)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset overview
print(f"Dataset Title: {metadata.name}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available **record sets**, their fields, and the Croissant `@id` for each. This will help you reference the fields precisely in your subsequent work.

In [ ]:
# List all available record sets and their fields, by @id
record_sets = list(dataset.record_sets)
print(f"Available record sets (by @id):")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name})")
    print()

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. Use the record set and field `@id`s from the overview above. This enables precise and consistent referencing for all subsequent steps.

In [ ]:
import pprint

# Assign the @id(s) of interest for the main record set(s) (replace these as appropriate if you wish to analyze others)
# We'll extract all record sets, but typically you pick the main table for analysis if there's more than one.
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for recset_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"  Columns in DataFrame (fields by @id):")
        for col in df.columns:
            print(f"   - {col}")
        display(df.head())
    else:
        print(f"  No records found for RecordSet @id {recset_id}\n")

# For demonstration, pick the first record set as the main analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We'll filter based on a numeric field, normalize it, and optionally group by a categorical field, all by referencing the Croissant `@id`.

**Instructions:**
- Choose a numeric field by its Croissant `@id` (see printed columns above).
- Choose an optional group/categorical field by `@id`.

In [ ]:
# Example field selection (update as needed!):
numeric_field_id = None
group_field_id = None

# Attempt to auto-select a numeric field if present
import numpy as np
main_df = dataframes.get(main_record_set_id, pd.DataFrame())
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break

# Try also finding a string column for grouping
for col in main_df.columns:
    if pd.api.types.is_string_dtype(main_df[col]) and col != numeric_field_id:
        group_field_id = col
        break

if numeric_field_id is None or main_df.empty:
    print("No numeric field was automatically found, or no records loaded. Please choose a valid numeric field @id from your data above.")
else:
    threshold = np.nanquantile(main_df[numeric_field_id], 0.75) # e.g., filter top 25% as an example
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (quartile example):")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized values for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}, mean of {numeric_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field or relationships between fields. Here we create a simple histogram for the chosen numeric field, and optionally a group comparison plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not main_df.empty:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        # Only plot if it's a categorical with not too many levels
        counts = main_df[group_field_id].value_counts()
        if len(counts) > 1 and len(counts) < 20:
            plt.figure(figsize=(8,5))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
- In this exploration with `mlcroissant`, you loaded the FAIR^2 mass spectrometry protein dataset and referenced all fields and record sets by Croissant `@id`.
- You reviewed available fields/entities, filtered and normalized numeric data, and visualized distributions.
- For a deeper analysis or ML applications, continue to reference all entities using their Croissant `@id` for full reproducibility and FAIRness.